In [ ]:
import google.generativeai as genai
import json
import uuid
import os
import time
from datetime import datetime

# 1. Cấu hình API Key
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY" # Điền API Key của bạn
genai.configure(api_key=GOOGLE_API_KEY)

# Sử dụng model bạn đang có (bạn thay lại đúng tên model 2.5 flash của bạn nhé)
model = genai.GenerativeModel('models/gemini-2.5-flash') 

# ==========================================
# CẤU HÌNH CHIẾN DỊCH SINH DỮ LIỆU
# ==========================================
RECORDS_PER_BATCH = 10  # Số dòng mỗi lần gọi (Không nên để quá 15 để tránh lỗi JSON do quá dài)
TOTAL_BATCHES = 100     # Số lần gọi API. 100 lần x 10 dòng = 1,000 bản ghi/lần chạy.
WAIT_TIME = 5           # Nghỉ 5 giây giữa các lần gọi để lách luật 15 Requests/Phút của Google

save_dir = r"D:\Data set\data transcript"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

print(f"Bắt đầu chiến dịch: Sinh {TOTAL_BATCHES * RECORDS_PER_BATCH} bản ghi.")
print(f"Dữ liệu sẽ được lưu tại: {save_dir}\n")

total_generated = 0

# 2. Vòng lặp tự động hóa
for batch_num in range(1, TOTAL_BATCHES + 1):
    print(f"Đang xử lý Batch {batch_num}/{TOTAL_BATCHES}...")
    
    prompt = f"""
    Act as a Synthetic Data Generator for a Data Engineering project.
    Generate EXACTLY {RECORDS_PER_BATCH} realistic B2C financial telesales call records in English.

    Requirements:
    1. The `call_transcript` must be realistic and "messy". Include interruptions, hesitations (umm, uh), background noise mentions, sudden hang-ups, or aggressive rejections.
    2. The `call_code` must be an array of EXACTLY 4 strings that summarize the chronological progression of the conversation (Stage 1 -> Stage 2 -> Stage 3 -> Stage 4).
       Choose from: ["GREETING", "QUALIFICATION", "PRODUCT_PITCH", "OBJECTION_HANDLING", "FEE_DISCUSSION", "NOT_INTERESTED", "CALL_BACK_SCHEDULED", "SUCCESSFUL_SALE", "SUDDEN_HANG_UP", "WRONG_PERSON", "VOICEMAIL", "ANGRY_CUSTOMER", "ESCALATION"]
    3. Return ONLY a valid JSON array of objects.

    Structure:
    {{
        "unique_id": "string",
        "call_transcript": "string",
        "call_code": ["Code1", "Code2", "Code3", "Code4"]
    }}
    """

    try:
        # Gọi API
        response = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                temperature=0.9 # Tăng temperature lên 0.9 để 1000 bản ghi không bị lặp kịch bản
            )
        )

        # Parse JSON
        dataset = json.loads(response.text)

        # Gắn UUID
        for record in dataset:
            record['unique_id'] = str(uuid.uuid4())

        # Lưu riêng từng batch ra 1 file (Chuẩn Data Lake: Small files into Raw Zone)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        file_name = f"transcript_batch{batch_num}_{timestamp}.json"
        file_path = os.path.join(save_dir, file_name)

        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, ensure_ascii=False, indent=4)

        total_generated += len(dataset)
        print(f" -> Thành công! Đã lưu file {file_name} ({len(dataset)} dòng). Tổng: {total_generated} dòng.")

    except Exception as e:
        print(f" -> [LỖI] Batch {batch_num} thất bại: {e}")
        # Nếu bị Google chặn do hết quota, dừng vòng lặp ngay để bảo toàn data đã sinh
        if "429" in str(e) or "quota" in str(e).lower():
            print("\nĐã chạm ngưỡng giới hạn miễn phí của Google. Tạm dừng chiến dịch!")
            break

    # Nghỉ ngơi giữa các loop để không bị cấm API
    if batch_num < TOTAL_BATCHES:
        time.sleep(WAIT_TIME)

print(f"\n🎉 HOÀN THÀNH! Đã sinh tổng cộng {total_generated} bản ghi thành công.")

In [ ]:
import google.generativeai as genai
import json
import uuid
import os
import time
from datetime import datetime

# 1. Cấu hình API Key
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY" # Điền API Key của bạn
genai.configure(api_key=GOOGLE_API_KEY)


model = genai.GenerativeModel('models/gemma-3-27b-it') 


RECORDS_PER_BATCH = 10
TOTAL_BATCHES = 1500     
WAIT_TIME = 3           

save_dir = r"D:\Data set\master_data"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

print(f"Bắt đầu chiến dịch: Sinh {TOTAL_BATCHES * RECORDS_PER_BATCH} bản ghi.")
total_generated = 0

# 2. Vòng lặp tự động hóa
for batch_num in range(1, TOTAL_BATCHES + 1):
    print(f"Đang xử lý Batch {batch_num}/{TOTAL_BATCHES}...")
    
    # PROMPT ĐÃ ĐƯỢC NÂNG CẤP
    prompt = f"""
    Act as an expert Conversational AI Data Generator. 
    Generate EXACTLY {RECORDS_PER_BATCH} highly realistic, multi-turn B2C financial telesales call transcripts in English.
    
    You must return ONLY a valid JSON array of objects representing a flat Master Data table. DO NOT include ```json or markdown tags.

    CRITICAL REQUIREMENTS FOR `call_transcript`:
    1. **Complex & Messy Verbal Dialogue**: The transcript must reflect highly realistic spoken words. Include filler words (um, uh, like), stuttering, sudden topic changes, overlapping speech, and abrupt verbal endings. DO NOT include background noise descriptions.
    2. **DATA CONSISTENCY (CRITICAL)**: The agent in the `call_transcript` MUST explicitly mention the EXACT `product_name`, `loan_amount`, and `interest_rate` that you generated in the fields for this specific record. 
       - Example: If `loan_amount` is 15000 and `interest_rate` is 12.5, the Agent MUST say something like: "We are pre-approving you for a $15,000 [product_name] at a rate of 12.5%..."
       - Do not let the agent offer different numbers than what is in the data fields.

    CRITICAL REQUIREMENTS FOR `call_code` (Exactly 4 strings per array):
    Must tell a psychological story using ONLY these exact values:
    - [STAGES]: "OPENING", "NEEDS_ANALYSIS", "PRODUCT_PITCH", "FEE_DISCUSSION", "COMPETITOR_COMPARISON", "OBJECTION_HANDLING", "CLOSING_NEGOTIATION"
    - [ATTITUDES]: "ENTHUSIASTIC_AGREEMENT", "ACTIVE_LISTENING", "PASSIVE_AGREEMENT", "CURIOUS_EXPLORATION", "APATHETIC_RESPONSE", "OVERWHELMED_CONFUSION", "ANGRY_OUTBURST", "SARCASTIC_MOCKERY", "SUSPICIOUS_PROBING", "ANNOYED_SIGHING", "DEFENSIVE_POSTURE"
    - [BEHAVIORS]: "FREQUENT_INTERRUPTIONS", "RUSHING_THE_CALL", "STALLING_FOR_TIME", "THREATENING_COMPLAINT", "DEMANDING_MANAGER", "INDECISIVE_FLIPPING"
    - [OUTCOMES]: "HARD_REJECTION", "SOFT_REJECTION", "DO_NOT_CALL_REQUEST", "FOLLOW_UP_EMAIL_REQUESTED", "WARM_LEAD", "SUCCESSFUL_SALE", "SUDDEN_HANG_UP"

    CRITICAL STRICT LOGICAL RULES (Must Follow):
    1. **Time vs. Outcome**: If `talk_time_seconds` < 60, transcript is very short, outcome MUST be "SUDDEN_HANG_UP" or "HARD_REJECTION". If > 300, it must be long, detailing fees, outcome is likely "SUCCESSFUL_SALE" or "WARM_LEAD".
    2. **History vs. Attitude**: If `previous_contact_count` >= 2, the customer is annoyed. Transcript MUST reflect frustration ("You guys called me yesterday!"), codes include "ANGRY_OUTBURST" or "DO_NOT_CALL_REQUEST".
    3. **Source vs. Receptiveness**: If `lead_source` is "Organic_Web_Form", they expect the call. If "Cold_Bought_List", they are defensive ("Where did you get my number?").
    4. **Age vs. Behavior**: If `age` > 60, include traits like asking to repeat things ("OVERWHELMED_CONFUSION"). If `age` < 30, they might rush ("RUSHING_THE_CALL").

    REQUIRED JSON SCHEMA (Flat Structure):
    [
      {{
        "customer_id": "string (e.g., CUST-1001)",
        "full_name": "string",
        "age": integer (22-70),
        "gender": "string (Male/Female/Other)",
        "phone_number": "string",
        "national_id": "string",
        "address": "string (City, State)",
        "employment_status": "string (Salaried/Self-employed/Unemployed/Retired)",
        "monthly_income": integer (3000-15000),
        "credit_score": integer (500-850),
        "is_existing_customer": boolean,
        
        "campaign_id": "string",
        "product_name": "string (e.g., Signature Credit Card, Personal Loan)",
        "lead_source": "string (Organic_Web_Form, Partner_Referral, Cold_Bought_List)",
        "decile_group": integer (1-10),
        "loan_amount": integer,
        "interest_rate": float,
        "previous_contact_count": integer (0-5),
        
        "call_id": "string",
        "agent_id": "string (e.g., AGT-204)",
        "call_timestamp": "string (ISO 8601)",
        "call_status": "Answered",
        "talk_time_seconds": integer (15-900),
        "call_code": ["Code1", "Code2", "Code3", "Code4"],
        "call_transcript": "string"
      }}
    ]
    """

    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                temperature=0.9, # Tăng nhẹ temp để hội thoại sáng tạo và đa dạng hơn
                top_p=0.95
            )
        )

        raw_text = response.text.strip()

        # DỌN DẸP DỮ LIỆU THÔ
        if raw_text.startswith("```json"):
            raw_text = raw_text.removeprefix("```json")
        if raw_text.startswith("```"):
            raw_text = raw_text.removeprefix("```")
        if raw_text.endswith("```"):
            raw_text = raw_text.removesuffix("```")
        
        raw_text = raw_text.strip()

        # Parse JSON
        dataset = json.loads(raw_text)

        # Gắn UUID
        for record in dataset:
            record['unique_id'] = str(uuid.uuid4())

        # Lưu file
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        file_name = f"transcript_batch{batch_num}_{timestamp}.json"
        file_path = os.path.join(save_dir, file_name)

        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, ensure_ascii=False, indent=4)

        total_generated += len(dataset)
        print(f" -> Thành công! Đã lưu {file_name} ({len(dataset)} dòng). Tổng: {total_generated} dòng.")

    except json.JSONDecodeError as json_err:
        print(f" -> [LỖI JSON] Batch {batch_num} bị lỗi định dạng. Model có thể đã sinh ra text thừa hoặc bị ngắt chuỗi.")
        # Lưu lại text lỗi để debug nếu cần
        with open(os.path.join(save_dir, f"error_batch{batch_num}.txt"), "w", encoding="utf-8") as err_file:
            err_file.write(raw_text)
            
    except Exception as e:
        print(f" -> [LỖI API/MẠNG] Batch {batch_num} thất bại: {e}")
        if "429" in str(e) or "quota" in str(e).lower():
            print("\nĐã chạm ngưỡng giới hạn. Tạm dừng chiến dịch!")
            break

    if batch_num < TOTAL_BATCHES:
        time.sleep(WAIT_TIME)

print(f"\n🎉 HOÀN THÀNH! Đã sinh tổng cộng {total_generated} bản ghi thành công.")

c:\Users\quoct\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\quoct\AppData\Local\Temp\ipykernel_11164\2240093899.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Bắt đầu chiến dịch: Sinh 5000 bản ghi.
Đang xử lý Batch 1/1000...
 -> Thành công! Đã lưu transcript_batch1_20260325_212152.json (5 dòng). Tổng: 5 dòng.
Đang xử lý Batch 2/1000...
 -> Thành công! Đã lưu transcript_batch2_20260325_212251.json (5 dòng). Tổng: 10 dòng.
Đang xử lý Batch 3/1000...
 -> Thành công! Đã lưu transcript_batch3_20260325_212354.json (5 dòng). Tổng: 15 dòng.
Đang xử lý Batch 4/1000...
 -> Thành công! Đã lưu transcript_batch4_20260325_212454.json (5 dòng). Tổng: 20 dòng.
Đang xử lý Batch 5/1000...
 -> Thành công! Đã lưu transcript_batch5_20260325_212555.json (5 dòng). Tổng: 25 dòng.
Đang xử lý Batch 6/1000...
 -> Thành công! Đã lưu transcript_batch6_20260325_212656.json (5 dòng). Tổng: 30 dòng.
Đang xử lý Batch 7/1000...
 -> Thành công! Đã lưu transcript_batch7_20260325_212758.json (5 dòng). Tổng: 35 dòng.
Đang xử lý Batch 8/1000...
 -> Thành công! Đã lưu transcript_batch8_20260325_212919.json (5 dòng). Tổng: 40 dòng.
Đang xử lý Batch 9/1000...
 -> Thành công! Đã lưu 

KeyboardInterrupt: 

In [2]:
import json
import glob
import csv
import os

# 1. Khai báo đường dẫn thư mục chứa các file JSON của bạn
folder_path = r"D:\Data set\upgrade prompt"

# Đường dẫn file CSV xuất ra
output_csv = os.path.join(folder_path, "all_transcripts_combined.csv")

# 2. Tìm tất cả các file có dạng transcript_*.json trong thư mục
file_pattern = os.path.join(folder_path, "transcript_*.json")
json_files = glob.glob(file_pattern)

all_data = []

print(f"Đã tìm thấy {len(json_files)} file JSON. Đang xử lý...")

# 3. Đọc dữ liệu từ từng file
for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            # Quét từng bản ghi trong file json
            for item in data:
                unique_id = item.get('unique_id', '')
                call_transcript = item.get('call_transcript', '')
                call_code = item.get('call_code', [])
                
                # Chuyển mảng call_code thành chuỗi, cách nhau bởi dấu phẩy
                if isinstance(call_code, list):
                    call_code_str = ", ".join(call_code)
                else:
                    call_code_str = str(call_code)
                
                # Thêm vào danh sách tổng
                all_data.append({
                    'unique_id': unique_id,
                    'call_transcript': call_transcript,
                    'call_code': call_code_str
                })
        except json.JSONDecodeError:
            print(f"Lỗi: Không thể đọc định dạng JSON của file {file}")

# 4. Ghi tất cả dữ liệu ra file CSV duy nhất
if all_data:
    # Dùng utf-8-sig để Excel mở không bị lỗi font tiếng Việt/ký tự đặc biệt
    with open(output_csv, 'w', encoding='utf-8-sig', newline='') as f:
        fieldnames = ['unique_id', 'call_transcript', 'call_code']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(all_data)
        
    print(f"✅ Hoàn tất! Đã lưu {len(all_data)} dòng dữ liệu vào file:")
    print(output_csv)
else:
    print("❌ Không tìm thấy dữ liệu để gộp.")

Đã tìm thấy 1156 file JSON. Đang xử lý...
✅ Hoàn tất! Đã lưu 17322 dòng dữ liệu vào file:
D:\Data set\upgrade prompt\all_transcripts_combined.csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import ast

# 1. Đọc dữ liệu từ file CSV đã gộp
csv_file_path = r"D:\Data set\all_transcripts_combined.csv"
df = pd.read_csv(csv_file_path)

# (Tùy chọn) Biến đổi chuỗi "nhãn1, nhãn2" trở lại thành danh sách (list) nếu cần
# df['call_code'] = df['call_code'].apply(lambda x: [label.strip() for label in x.split(',')])

print(f"Tổng số mẫu dữ liệu: {len(df)}")

# 2. Chia dữ liệu
# Bước A: Tách tập Train (80%) và tập Tạm thời (20%)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# Bước B: Chia tập Tạm thời làm đôi để lấy Validation (10%) và Test (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Số lượng tập Train: {len(train_df)} ({(len(train_df)/len(df))*100:.0f}%)")
print(f"Số lượng tập Validation: {len(val_df)} ({(len(val_df)/len(df))*100:.0f}%)")
print(f"Số lượng tập Test: {len(test_df)} ({(len(test_df)/len(df))*100:.0f}%)")

# 3. Lưu ra 3 file CSV riêng biệt để chuẩn bị train model
train_df.to_csv(r"D:\Data set\train.csv", index=False, encoding='utf-8-sig')
val_df.to_csv(r"D:\Data set\valid.csv", index=False, encoding='utf-8-sig')
test_df.to_csv(r"D:\Data set\test.csv", index=False, encoding='utf-8-sig')

print("✅ Đã lưu thành công 3 file: train.csv, valid.csv, test.csv")